# Session 1 — Register to Unity Catalog & Deploy Serving Endpoint

**Goal:** Take the best model from our MLflow experiment and:
1. Register it in the Unity Catalog Model Registry
2. Assign it the `champion` alias
3. Deploy it as a real-time serving endpoint
4. Query it with a sample customer record

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ --quiet

## Provide your Best Run ID
1. Paste the value of your **best_run_id** in the notebook parameter at the top of the page.

In [0]:

dbutils.widgets.text("best_run_id", "", "Best Run ID (from 04_mlflow_experiment)")

best_run_id = dbutils.widgets.get("best_run_id")

if not best_run_id:
    # Try to get from task values (when running as a job)
    try:
        best_run_id = dbutils.jobs.taskValues.get(taskKey="train_models", key="best_run_id")
    except Exception:
        raise ValueError("Please enter the best_run_id from 04_mlflow_experiment.ipynb")

## MLflow model registry
MLflow can utilize the registry of your choice.  By default in Databricks, MLflow will register models in the workspace files space.  In the following cell, we instruct MLflow to register the model with Unity Catalog so that we can utilize Unity Catalog's full governance for the model.

In [0]:
import mlflow
from mlflow import MlflowClient

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Instantiate a MLflow client to be used later
client = MlflowClient()

model_name = f"{catalog}.{schema}.churn_classifier"
print(f"Model name  : {model_name}")
print(f"Best run ID : {best_run_id}")

## Step 1: Register the Model in Unity Catalog

Registering a model in Unity Catalog:
- Creates a versioned, governed model artifact
- Links the model back to the MLflow run (lineage!)
- Enables you to assign human-readable aliases like `@champion`

In [0]:
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)

print(f"✓ Registered: {registered.name}")
print(f"  Version   : {registered.version}")
print(f"  Status    : {registered.status}")

In [0]:
# Assign the 'champion' alias — this is what Model Serving and batch jobs will reference
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered.version,
)
print(f"✓ Alias 'champion' set on version {registered.version}")
print(f"\nModel URI with alias: models:/{model_name}@champion")

### Navigate to Unity Catalog

1. Click **Catalog** in the left sidebar
2. Navigate: `<catlog>` → `<your_schema>` → **Models** → `churn_classifier`
3. Observe:
   - Version history
   - The `champion` alias tag
4. Select version 1 of the model.
5. Observe:
   - **Lineage** tab — links back to your MLflow run
   - **Description** — empty now, but in production you'd add a model card here

## Step 2: Create a Model Serving Endpoint

Databricks Model Serving creates a managed REST endpoint that:
- Auto-scales from zero (no cost when idle)
- Handles versioning and traffic routing
- Logs every request/response to an **AI Gateway inference table** (used for monitoring in Session 2)
- **Rate-limits requests** via AI Gateway (60 calls/min per user)

The inference table will be created automatically at `workshop.<your_schema>.churn_payload`.

> This takes **5-10 minutes** to reach the Ready state.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    EndpointCoreConfigInput,
    ServedModelInput,
)

w = WorkspaceClient()
endpoint_name = f"churn_{safe_username}"

print(f"Creating endpoint: {endpoint_name}")
print("This will take 5-10 minutes...")

w.serving_endpoints.create_and_wait(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        served_models=[
            ServedModelInput(
                model_name=model_name,
                model_version=str(registered.version),
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix="churn",
            enabled=True,
        ),
        rate_limits=[
            AiGatewayRateLimit(
                calls=60,
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
    ),
)
print(f"✓ Endpoint '{endpoint_name}' is Ready!")
print(f"  Inference table: {catalog}.{schema}.churn_payload")
print(f"  Rate limit     : 60 requests/min per user")

### Navigate to Model Serving

1. Click **Serving** in the left sidebar
2. Find `churn_<safe_username>`
3. When the status shows **Ready**, proceed to the next cell

Observe the endpoint configuration:
- Served model version
- Scale-to-zero setting
- Use → Query UI (try it!)

## Step 3: Test


#### Python function
Load the model from Unity Catalog as a Python function and run a test inference

In [0]:
import mlflow
import time
import numpy as np
import pandas as pd

start = time.time()
# Load the model from the Unity Catalog registry using the @champion alias
model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")
print(f"Model loaded in {time.time() - start:.1f}s")

# Also test a prediction
sample_input = pd.DataFrame([{
    "gender":           "Female",
    "SeniorCitizen":    "0",
    "Partner":          "No",
    "Dependents":       "No",
    "tenure":           np.int32(1),
    "PhoneService":     "Yes",
    "MultipleLines":    "No",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "No",
    "OnlineBackup":     "No",
    "DeviceProtection": "No",
    "TechSupport":      "No",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Electronic check",
    "MonthlyCharges":   85.7,
    "TotalCharges":     85.7,
}])
model.predict(sample_input)

#### REST Interface
Use REST to make an inference

In [0]:
import requests, json

# Get workspace URL and auth headers (works with OAuth, PAT, or any auth method)
w_client = WorkspaceClient()
workspace_url = w_client.config.host
headers = {"Content-Type": "application/json"}
headers.update(w_client.config.authenticate())

# Sample customer: new month-to-month customer, high monthly charges — high churn risk
sample_customer = {
    "dataframe_records": [{
        "customerID":       "NEW_CUSTOMER_001",
        "gender":           "Female",
        "SeniorCitizen":    0,
        "Partner":          "No",
        "Dependents":       "No",
        "tenure":           1,
        "PhoneService":     "Yes",
        "MultipleLines":    "No",
        "InternetService":  "Fiber optic",
        "OnlineSecurity":   "No",
        "OnlineBackup":     "No",
        "DeviceProtection": "No",
        "TechSupport":      "No",
        "StreamingTV":      "Yes",
        "StreamingMovies":  "Yes",
        "Contract":         "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod":    "Electronic check",
        "MonthlyCharges":   85.7,
        "TotalCharges":     85.7,
    }]
}

response = requests.post(
    f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations",
    headers=headers,
    json=sample_customer,
    timeout=30,
)

result = response.json()
print(f"Status: {response.status_code}")
print(f"Prediction: {json.dumps(result, indent=2)}")

## Bonus: LLM-Powered Churn Explanation

The churn model returns a prediction — but **why** did this customer churn?

We can create a second serving endpoint for the **Foundation Model API** (`databricks-meta-llama-3-3-70b-instruct`) to generate a human-readable explanation.

This demonstrates:
- Creating a **legacy AI Gateway** endpoint programmatically — the same governance layer used in Step 2, embedded directly in Model Serving
- **Guardrails** configured at the gateway layer: PII blocking on inputs, safety filtering on outputs
- **Foundation Model API** (pay-per-token, no provisioning needed)
- How GenAI augments traditional ML in production workflows

> **Guardrails are applied at the gateway layer** — all apps sharing this endpoint get the same safety policy without changing application code.

In [ ]:
from databricks.sdk.service.serving import (
    ServedEntityInput,
    AiGatewayGuardrails,
    AiGatewayGuardrailParameters,
    AiGatewayGuardrailPiiBehavior,
    AiGatewayGuardrailPiiBehaviorBehavior,
)

llm_endpoint_name = f"{safe_username}_llm"

print(f"Creating LLM endpoint: {llm_endpoint_name}")
print("This may take a few minutes...")

try:
    w.serving_endpoints.get(name=llm_endpoint_name)
    print(f"Endpoint '{llm_endpoint_name}' already exists — skipping creation.")
except Exception:
    w.serving_endpoints.create_and_wait(
        name=llm_endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name="databricks-meta-llama-3-3-70b-instruct",
                    scale_to_zero_enabled=True,
                )
            ]
        ),
        ai_gateway=AiGatewayConfig(
            inference_table_config=AiGatewayInferenceTableConfig(
                catalog_name=catalog,
                schema_name=schema,
                table_name_prefix="llm",
                enabled=True,
            ),
            rate_limits=[
                AiGatewayRateLimit(
                    calls=10,
                    key=AiGatewayRateLimitKey.ENDPOINT,
                    renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                )
            ],
            guardrails=AiGatewayGuardrails(
                input=AiGatewayGuardrailParameters(
                    pii=AiGatewayGuardrailPiiBehavior(
                        behavior=AiGatewayGuardrailPiiBehaviorBehavior.BLOCK,
                    ),
                    safety=True,
                ),
                output=AiGatewayGuardrailParameters(
                    safety=True,
                ),
            ),
        ),
    )
    print(f"✓ LLM endpoint '{llm_endpoint_name}' is Ready!")
    print(f"  Model          : databricks-meta-llama-3-3-70b-instruct")
    print(f"  Inference table: {catalog}.{schema}.llm_payload")
    print(f"  Rate limit     : 10 requests/min per endpoint")
    print(f"  Guardrails     : PII blocking (input) + safety filtering (input & output)")

In [ ]:
# Query the LLM through the legacy AI Gateway endpoint
# Extract the prediction from the churn endpoint response
prediction = result.get("predictions", ["unknown"])[0]

# Build a concise customer summary for the LLM
customer = sample_customer["dataframe_records"][0]
customer_lines = "\n".join(
    f"  {k}: {v}" for k, v in customer.items() if k != "customerID"
)

prompt = f"""A customer churn prediction model predicted: {prediction}

Customer profile:
{customer_lines}

In 2-3 sentences, explain the most likely reasons for this prediction based on
the customer profile, and suggest one specific retention action."""


llm_response = requests.post(
    f"{workspace_url}/serving-endpoints/{llm_endpoint_name}/invocations",
    headers=headers,
    json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200,
    },
    timeout=30,
)

llm_result = llm_response.json()
explanation = llm_result["choices"][0]["message"]["content"]

print(f"Churn prediction : {prediction}")
print(f"LLM endpoint     : {llm_endpoint_name}")
print(f"\nLLM explanation:\n{explanation}")

### Legacy AI Gateway — What We Just Enabled

The endpoint above uses the **legacy AI Gateway**, embedded directly in Model Serving. All governance features are available today via API:

| Feature | Available |
|---|---|
| Inference table logging | ✅ Yes |
| Rate limiting | ✅ Yes |
| PII detection & blocking | ✅ Yes |
| Toxicity / safety guardrails | ✅ Yes |

> Both endpoints in this notebook (`churn_<username>` and `<username>_llm`) use the same gateway configuration pattern — giving you a consistent governance layer across custom ML models and Foundation Model API calls.

## Summary

✅ **Session 1 complete!** You have:

1. Explored the Telco dataset and understood its characteristics
2. Refactored a messy notebook into a modular, testable package
3. Trained 3 model types and compared them in MLflow
4. Registered the best model in Unity Catalog with a `@champion` alias
5. Deployed a rate-limited serving endpoint with AI Gateway
6. Made a live prediction via REST API
7. Generated an LLM explanation using Foundation Model API

**Before Session 2**, review [06_production_checklist.ipynb]($./06_production_checklist) to see what's still missing.
